In [8]:
import os
import pandas as pd
import statsmodels.api as sm
from statsmodels.sandbox.regression.gmm import IV2SLS
import numpy as np

os.chdir('/Users/fogellmcmuffin/Documents/thesis/_replication/')    # Working dir

pd.set_option('display.max_rows', 10)
pd.set_option('display.max_columns', None)

vintage_trim = pd.read_csv('cleaned_data/vintage_trim.csv')

mean_spf_trim = pd.read_csv('cleaned_data/mean_spf_trim.csv')
ind_spf_trim = pd.read_csv('cleaned_data/ind_spf_trim.csv')

mean_spf_trim['DATE'] = pd.to_datetime(mean_spf_trim['DATE'])
ind_spf_trim['DATE'] = pd.to_datetime(ind_spf_trim['DATE'])

ind_spf_trim = ind_spf_trim.dropna(subset='r_t1')

In [10]:
### OLS ###
def control(df, v, end_date):
    df = df.loc[(df['DATE'] >= '1981-12-31') & (df['DATE'] <= end_date)]
    v = v.loc[(v['DATE'] >= '1981-12-31') & (v['DATE'] <= end_date)]
    revisions = df[f'r_t3']
    L_pi = ((v['t']/400 + 1)*(v['t1']/400 + 1)*(v['t2']/400 + 1)*(v['t3']/400 + 1)-1)
    L_pi = L_pi.shift(1)
    x = np.column_stack((L_pi, revisions))
    x = sm.add_constant(x)
    errors = df[f'e_t3']
    initial = sm.OLS(errors[4:], x[4:]).fit()
    regs = initial.get_robustcov_results(cov_type='HAC', maxlags=None)
    return regs

date = '2020-06-30'
regm = control(mean_spf_trim, vintage_trim, date)

regm.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                   e_t3   R-squared:                       0.012
Model:                            OLS   Adj. R-squared:                 -0.002
Method:                 Least Squares   F-statistic:                    0.5010
Date:                Fri, 03 May 2024   Prob (F-statistic):              0.607
Time:                        16:28:06   Log-Likelihood:                 481.80
No. Observations:                 151   AIC:                            -957.6
Df Residuals:                     148   BIC:                            -948.5
Df Model:                           2                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0048      0.004     -1.232      0.220      -0.012       0.003
x1             0.0067      0.009      0.759      0.449      -0.011       0.024
x2             0.1607      0.192      0.838      0.404      -0.218       0.540
==============================================================================
Omnibus:                       11.223   Durbin-Watson:                   0.663
Prob(Omnibus):                  0.004   Jarque-Bera (JB):               20.091
Skew:                          -0.318   Prob(JB):                     4.34e-05
Kurtosis:                       4.670   Cond. No.                         266.
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using None lags and without small sample correction
"""

In [13]:
def measurement_err(df, end_date):
    df = df.loc[(df['DATE'] >= '1981-12-31') & (df['DATE'] <= end_date)]
    revisions = df[f'r_t3']
    revisions = sm.add_constant(revisions)
    errors = df[f'e_t2']
    initial = sm.OLS(errors, revisions).fit()
    regs = initial.get_robustcov_results(cov_type='HAC', maxlags=None)
    return regs


def measurement_err_ind(df, end_date):
    df = df.loc[(df['DATE'] >= '1981-12-31') & (df['DATE'] <= end_date)]
    revisions = df[f'r_t3']
    revisions = sm.add_constant(revisions)
    errors = df[f'e_t2']
    regs = sm.OLS(errors, revisions).fit(cov_type='cluster', cov_kwds = {"groups":df['ID']})
    return regs


measurement_err_ind(ind_spf_trim, '2020-06-30').summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                   e_t2   R-squared:                       0.022
Model:                            OLS   Adj. R-squared:                  0.022
Method:                 Least Squares   F-statistic:                     65.94
Date:                Sun, 05 May 2024   Prob (F-statistic):           2.78e-13
Time:                        20:48:08   Log-Likelihood:                 12110.
No. Observations:                3775   AIC:                        -2.422e+04
Df Residuals:                    3773   BIC:                        -2.420e+04
Df Model:                           1                                         
Covariance Type:              cluster                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0016      0.000     -5.712      0.000      -0.002      -0.001
r_t3          -0.2216      0.027     -8.121      0.000      -0.275      -0.168
==============================================================================
Omnibus:                      432.204   Durbin-Watson:                   0.383
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             1295.414
Skew:                          -0.602   Prob(JB):                    5.06e-282
Kurtosis:                       5.605   Cond. No.                         150.
==============================================================================

Notes:
[1] Standard Errors are robust to cluster correlation (cluster)
"""